# 🚀 SAM-AI R1: Autonomous ARC-AGI & Multi-Domain GRPO Training

**Architecture:** DeepSeek-R1 Distill (System 2 Thinking Engine)  
**Optimization:** Group Relative Policy Optimization (GRPO) with Deterministic RLVR  
**Hardware Target:** Optimized for NVIDIA RTX Pro 6000 (48GB VRAM) & Scalable GPU Clusters  
**Domains:** ARC-AGI Visual-Spatial Transformations + Verifiable Mathematical Reasoning

---

In [ ]:
# ==============================================================================
# 1. Environment & Dependency Conflict Resolution
# ==============================================================================
import subprocess
import sys
import os

print("[*] Checking and resolving Kaggle pre-installed environment conflicts...")
# Fix Kaggle's incompatible torchao==0.10.0 that breaks modern PEFT:
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

# Install / upgrade required packages cleanly
pkgs = ["trl>=0.15.0", "peft>=0.12.0", "transformers>=4.48.0", "accelerate>=1.2.0", "datasets", "bitsandbytes"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

print("[✓] Environment sanitized: torchao conflict resolved. Modern TRL and PEFT ready.")


In [ ]:
# ==============================================================================
# 2. Hardware Diagnostic & VRAM Detection
# ==============================================================================
import torch

print("=" * 70)
print("  SAM-AI HARDWARE ACCELERATOR DIAGNOSTIC")
print("=" * 70)

if not torch.cuda.is_available():
    print("[!] CUDA is not available. Please enable GPU accelerator in Notebook Settings.")
    gpu_name = "CPU"
    vram_gb = 0.0
    bf16_supported = False
else:
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    bf16_supported = torch.cuda.is_bf16_supported()
    print(f"[✓] Detected GPU: {gpu_name}")
    print(f"[✓] Total VRAM:   {vram_gb:.2f} GB")
    print(f"[✓] bfloat16:     {'Supported' if bf16_supported else 'Not Supported'}")
    print(f"[✓] Device Count: {torch.cuda.device_count()}")
print("=" * 70)


In [ ]:
# ==============================================================================
# 3. Dynamic Model Architecture & Hyperparameter Selection
# ==============================================================================
# Scalable configuration:
# - If RTX Pro 6000 (48GB) or A100: Use DeepSeek-R1-Distill-Qwen-14B
# - If T4 / P100 (16GB): Use DeepSeek-R1-Distill-Qwen-1.5B (or 7B in 4-bit)
if vram_gb >= 40.0:
    MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"
    LOAD_4BIT = False
    BATCH_SIZE = 2
    GROUP_SIZE = 4
    MAX_COMPLETION_LENGTH = 1024
    GRAD_ACCUM_STEPS = 4
    print(f"[*] High-Capacity Workstation detected ({gpu_name}): Provisioning 14B Reasoning Engine.")
elif vram_gb >= 14.0:
    MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    LOAD_4BIT = False
    BATCH_SIZE = 2
    GROUP_SIZE = 4
    MAX_COMPLETION_LENGTH = 768
    GRAD_ACCUM_STEPS = 4
    print(f"[*] Standard Cloud GPU detected ({gpu_name}): Provisioning 1.5B Reasoning Engine.")
else:
    MODEL_ID = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
    LOAD_4BIT = True
    BATCH_SIZE = 1
    GROUP_SIZE = 2
    MAX_COMPLETION_LENGTH = 512
    GRAD_ACCUM_STEPS = 2
    print(f"[*] Resource-Constrained environment: Provisioning 1.5B with 4-bit quantization.")

LEARNING_RATE = 5e-6
NUM_TRAIN_EPOCHS = 2
OUTPUT_DIR = "/kaggle/working/sam_ai_r1_arc_trained"
print(f"[*] Foundation Model: {MODEL_ID}")
print(f"[*] GRPO Generations per Prompt: {GROUP_SIZE}")
print(f"[*] Max Reasoning Length: {MAX_COMPLETION_LENGTH} tokens")


In [ ]:
# ==============================================================================
# 4. ARC-AGI Procedural Synthesis & Verifiable Multi-Domain Dataset
# ==============================================================================
import random
import json
from datasets import Dataset

def format_grid(grid):
    """Converts a 2D grid into standard ARC string representation."""
    return "\n".join(" ".join(str(c) for c in row) for row in grid)

def rotate_90(grid):
    """Rotates a 2D grid 90 degrees clockwise."""
    return [list(row) for row in zip(*grid[::-1])]

def reflect_h(grid):
    """Reflects a 2D grid horizontally."""
    return [row[::-1] for row in grid]

def reflect_v(grid):
    """Reflects a 2D grid vertically."""
    return grid[::-1]

def color_remap(grid, old_c, new_c):
    """Substitutes old color with new color in the grid."""
    return [[new_c if c == old_c else c for c in row] for row in grid]

def generate_arc_dataset(n_samples=600):
    """Generates verifiable ARC-AGI grid transformation tasks."""
    random.seed(42)
    arc_tasks = []
    for _ in range(n_samples):
        h = random.randint(3, 5)
        w = random.randint(3, 5)
        colors = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
        used_colors = random.sample(colors, random.randint(2, 4))
        input_grid = [[random.choice(used_colors) for _ in range(w)] for _ in range(h)]
        
        rule_type = random.choice(["rotate_90", "rotate_180", "reflect_h", "reflect_v", "color_shift"])
        if rule_type == "rotate_90":
            output_grid = rotate_90(input_grid)
            task_desc = "Rotate the input grid 90 degrees clockwise."
        elif rule_type == "rotate_180":
            output_grid = rotate_90(rotate_90(input_grid))
            task_desc = "Rotate the input grid 180 degrees."
        elif rule_type == "reflect_h":
            output_grid = reflect_h(input_grid)
            task_desc = "Reflect the input grid horizontally (left-right flip)."
        elif rule_type == "reflect_v":
            output_grid = reflect_v(input_grid)
            task_desc = "Reflect the input grid vertically (upside-down flip)."
        else:
            c_src, c_dst = random.sample(used_colors, 2)
            output_grid = color_remap(input_grid, c_src, c_dst)
            task_desc = f"Replace every occurrence of color {c_src} with color {c_dst}."
        
        prompt = (
            f"[ARC-AGI Task] {task_desc}\n\n"
            f"Input Grid ({h}x{w}):\n{format_grid(input_grid)}\n\n"
            f"Determine the output grid step-by-step using <think>...</think> tags.\n"
            f"Output your final transformed grid inside <answer>...</answer> tags."
        )
        arc_tasks.append({
            "prompt": prompt,
            "answer": format_grid(output_grid),
            "domain": "arc_spatial"
        })
    return arc_tasks

def generate_math_dataset(n_samples=600):
    """Generates verifiable arithmetic, algebraic, and multi-step word problems."""
    random.seed(101)
    math_tasks = []
    for _ in range(n_samples):
        kind = random.choice(["arithmetic", "algebra", "modulo"])
        if kind == "arithmetic":
            a, b, c = random.randint(12, 99), random.randint(11, 45), random.randint(3, 9)
            ans = (a + b) * c
            prompt = f"Evaluate step-by-step using <think>...</think>: ({a} + {b}) * {c} = ? Put final integer in <answer>...</answer>."
        elif kind == "algebra":
            x = random.randint(3, 30)
            m = random.randint(3, 8)
            k = random.randint(10, 50)
            rhs = m * x + k
            ans = x
            prompt = f"Solve for x step-by-step: {m}x + {k} = {rhs}. Put the integer value of x inside <answer>...</answer>."
        else:
            val = random.randint(100, 999)
            mod = random.choice([7, 11, 13, 17])
            ans = val % mod
            prompt = f"Calculate the remainder of {val} divided by {mod} step-by-step using <think> tags. Put the remainder inside <answer>...</answer>."
        math_tasks.append({
            "prompt": prompt,
            "answer": str(ans),
            "domain": "verifiable_math"
        })
    return math_tasks

print("[*] Generating procedural ARC-AGI and Verifiable Math corpora...")
arc_data = generate_arc_dataset(600)
math_data = generate_math_dataset(600)
all_problems = arc_data + math_data
random.seed(777)
random.shuffle(all_problems)

train_dataset = Dataset.from_list([
    {
        "prompt": [
            {"role": "system", "content": "You are SAM-AI, a sovereign reasoning intelligence. Solve problems with rigorous System 2 deliberation using <think>...</think> tags, and enclose your final answer in <answer>...</answer> tags."},
            {"role": "user", "content": p["prompt"]}
        ],
        "answer": p["answer"],
        "domain": p["domain"]
    }
    for p in all_problems
])
print(f"[✓] Training dataset synthesized: {len(train_dataset)} verifiable tasks ({len(arc_data)} ARC grids, {len(math_data)} Math derivations).")


In [ ]:
# ==============================================================================
# 5. RLVR Deterministic Reward Verifiers (Rule-Based Ground Truth)
# ==============================================================================
import re

def extract_answer_tag(text):
    """Extracts the content enclosed inside <answer>...</answer>."""
    match = re.search(r"<answer>(.*?)</answer>", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return None

def sam_rlvr_reward_fn(completions, answer=None, domain=None, **kwargs):
    """
    Computes composite deterministic rewards across reasoning quality and accuracy:
    - Format Reward (0.5): Genuine <think>...</think> + <answer>...</answer> structure
    - Accuracy Reward (1.0): Exact objective match of the derived solution
    - Partial Match Reward (0.25): Close numerical or dimension match
    """
    rewards = []
    n_comp = len(completions)
    ans_list = answer if answer is not None else []
    dom_list = domain if domain is not None else []
    if len(ans_list) > 0 and len(ans_list) != n_comp:
        rep = max(1, n_comp // len(ans_list))
        ans_list = [a for a in ans_list for _ in range(rep)][:n_comp]
        dom_list = [d for d in dom_list for _ in range(rep)][:n_comp]
    elif len(ans_list) == 0:
        ans_list = [""] * n_comp
        dom_list = [""] * n_comp

    for i, completion in enumerate(completions):
        target = ans_list[i] if i < len(ans_list) else ""
        d = dom_list[i] if i < len(dom_list) else ""
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion and isinstance(completion[-1], dict) else str(completion)
        else:
            text = str(completion)
        r = 0.0
        
        # 1. System 2 Thinking Format Reward
        has_think = ("<think>" in text and "</think>" in text)
        has_answer = ("<answer>" in text and "</answer>" in text)
        
        if has_think:
            think_content = text.split("</think>")[0].replace("<think>", "").strip()
            if len(think_content) >= 30:
                r += 0.35  # Sustained chain-of-thought
        if has_answer:
            r += 0.15      # Structured output tags
        
        extracted = extract_answer_tag(text)
        target_clean = str(target).strip()
        
        # 2. Objective Ground-Truth Accuracy Reward
        if d == "arc_spatial":
            # Normalize whitespace for 2D grids
            norm_target = re.sub(r"\s+", " ", target_clean)
            if extracted:
                norm_pred = re.sub(r"\s+", " ", extracted.strip())
                if norm_pred == norm_target:
                    r += 1.0
                elif len(norm_pred) > 0 and (norm_pred in norm_target or norm_target in norm_pred):
                    r += 0.35
            else:
                norm_text = re.sub(r"\s+", " ", text)
                if norm_target in norm_text:
                    r += 0.5
        else:
            if extracted and extracted == target_clean:
                r += 1.0
            else:
                nums = re.findall(r"\b\d+\b", text)
                if target_clean in nums:
                    last_idx = text.rfind(target_clean)
                    if last_idx > len(text) * 0.5:
                        r += 0.7
                    else:
                        r += 0.3
        rewards.append(r)
    return rewards

print("[✓] Deterministic RLVR Verifier registered.")


In [ ]:
# ==============================================================================
# 6. Model & LoRA Adapter Initialization (Bulletproof Device Placement)
# ==============================================================================
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

print(f"[*] Loading tokenizer: {MODEL_ID}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"[*] Loading weights for {MODEL_ID}...")
# Use explicit device_map={"": 0} on single GPU to prevent any CPU meta-device offloading
compute_dtype = torch.bfloat16 if bf16_supported else torch.float16

if LOAD_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map={"": 0},
        trust_remote_code=True,
    )
else:
    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=compute_dtype,
        device_map={"": 0},
        trust_remote_code=True,
    )

# Configure Low-Rank Adaptation (LoRA) for efficient System 2 reasoning updates
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(base_model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"[✓] LoRA Adapter Injected successfully!")
print(f"[✓] Trainable Parameters: {trainable:,} / {total:,} ({100 * trainable / total:.2f}%)")


In [ ]:
# ==============================================================================
# 7. Group Relative Policy Optimization (GRPO) Reinforcement Learning
# ==============================================================================
import os
import inspect
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    num_generations=GROUP_SIZE,
    max_completion_length=MAX_COMPLETION_LENGTH,
    learning_rate=LEARNING_RATE,
    logging_steps=5,
    save_steps=50,
    bf16=bf16_supported,
    fp16=not bf16_supported,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    report_to="none",
)

# Dynamically detect if modern TRL requires 'processing_class' or 'tokenizer':
grpo_init_kwargs = {}
sig = inspect.signature(GRPOTrainer.__init__)
if 'processing_class' in sig.parameters:
    grpo_init_kwargs['processing_class'] = tokenizer
elif 'tokenizer' in sig.parameters:
    grpo_init_kwargs['tokenizer'] = tokenizer

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    reward_funcs=sam_rlvr_reward_fn,
    **grpo_init_kwargs
)

print("=" * 70)
print("  STARTING GRPO REINFORCEMENT LEARNING RUN")
print("=" * 70)
trainer.train()

print("\n[*] Training run finished. Saving calibrated model artifacts...")
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"[🎉] Checkpoint preserved at: {OUTPUT_DIR}")

# ==============================================================================
# 8. Publish Live to Hugging Face Hub
# ==============================================================================
HF_REPO = 'Samrish2009/SAM-AI-Reasoning-14B'
HF_TOKEN = os.environ.get('HF_TOKEN', '')
try:
    print(f'[*] Publishing live checkpoint directly to Hugging Face Hub: {HF_REPO}...')
    trainer.model.push_to_hub(HF_REPO, token=HF_TOKEN)
    tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
    print(f'[✓] MODEL IS OFFICIALLY LIVE ON THE WEB: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'[Notice] Hub push error: {e}')


In [ ]:
# ==============================================================================
# 8. Test Inference & System 2 Verification Demo
# ==============================================================================
test_sample = arc_data[0]["prompt"]
print("[*] Testing SAM-AI R1 on held-out ARC spatial task:\n")
print(test_sample)

inputs = tokenizer(
    f"<|im_start|>system\nYou are SAM-AI, a sovereign reasoning intelligence. Think step-by-step using <think> tags and output final answer in <answer> tags.<|im_end|>\n<|im_start|>user\n{test_sample}<|im_end|>\n<|im_start|>assistant\n",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.6,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("=" * 70)
print("  SAM-AI R1 REASONING TRACE & GENERATED SOLUTION")
print("=" * 70)
print(response)
print("=" * 70)
